In [5]:
from pyspark.sql import functions as F

# 1. Silver transform
BRONZE = "tip_bronze"
SILVER = "tip_silver"

df_tip = spark.table(BRONZE)

StatementMeta(, c1be0e87-afdc-492e-bfed-447245c6644f, 7, Finished, Available, Finished, False)

In [6]:
base_cols = [
    "user_id",
    "business_id",
    "text",
    "date",
    "compliment_count",
    "_ingest_ts",
    "_source_file",
    "_batch_id"
]

df_tip_silver = df_tip.select(*base_cols)

StatementMeta(, c1be0e87-afdc-492e-bfed-447245c6644f, 8, Finished, Available, Finished, False)

In [7]:
df_tip_silver = (
    df_tip_silver
    .withColumn("text", F.trim(F.col("text")))
    .withColumn("compliment_count", F.coalesce(F.col("compliment_count"), F.lit(0)).cast("int"))
    
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("business_id").isNotNull())
    .filter(F.col("date").isNotNull())
    .filter(F.col("compliment_count") >= 0)

    .dropDuplicates(["user_id", "business_id", "text", "date"])
)

StatementMeta(, c1be0e87-afdc-492e-bfed-447245c6644f, 9, Finished, Available, Finished, False)

In [8]:
(
    df_tip_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER)
)

StatementMeta(, c1be0e87-afdc-492e-bfed-447245c6644f, 10, Finished, Available, Finished, False)

In [9]:
df_tip_silver.printSchema()
print("tip_silver rows: ", spark.table("tip_silver").count())

StatementMeta(, c1be0e87-afdc-492e-bfed-447245c6644f, 11, Finished, Available, Finished, False)

root
 |-- user_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- text: string (nullable = true)
 |-- date: string (nullable = true)
 |-- compliment_count: integer (nullable = false)
 |-- _ingest_ts: date (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



In [11]:
mssparkutils.notebook.exit("SUCCESS")

StatementMeta(, c1be0e87-afdc-492e-bfed-447245c6644f, 13, Finished, Available, Finished, False)

ExitValue: SUCCESS

In [15]:
#2. Data Quality Check

df_tip_silver_DQ = spark.table("tip_silver")

print("tip_silver rows = ", df_tip_silver_DQ.count())

dup = (
    df_tip_silver_DQ.groupBy("user_id", "business_id", "text", "date")
    .count()
    .filter(F.col("count") > 1)
    .filter(F.length(F.col("text")) > 0)
    .count()
)
print("duplicate tip rows = ", dup)

df_tip_silver_DQ.select(
    F.sum(F.col("user_id").isNull().cast("int")).alias("null_user_id"),
    F.sum(F.col("business_id").isNull().cast("int")).alias("null_business_id"),
    F.sum(F.col("date").isNull().cast("int")).alias("null_date"),
    F.sum(F.col("compliment_count").isNull().cast("int")).alias("null_compliment_count")
).show()

negative_compliment = df_tip_silver_DQ.filter(F.col("compliment_count") < 0).count()
print("negative compliment_count rows = ", negative_compliment)


StatementMeta(, c1be0e87-afdc-492e-bfed-447245c6644f, 17, Finished, Available, Finished, False)

tip_silver rows =  908848
duplicate tip rows =  0
+------------+----------------+---------+---------------------+
|null_user_id|null_business_id|null_date|null_compliment_count|
+------------+----------------+---------+---------------------+
|           0|               0|        0|                    0|
+------------+----------------+---------+---------------------+

negative compliment_count rows =  0
